# 02 - STL10 Experiments

Notebook này chỉ load feature `.npy` đã được trích xuất từ backbone MoCo-v2 và chạy tái hiện thực nghiệm chính trên STL-10.


In [1]:
import sys
import numpy as np
import pandas as pd
import torch

sys.path.append('..')
from src.utils import get_device, load_feature_file, set_seed, train_clustering

set_seed(42)
device = get_device()
X, y = load_feature_file('../data/stl10_moco_features.npy', '../data/stl10_labels.npy', device=device)
print('[INFO] STL-10 features:', tuple(X.shape), '| labels:', tuple(y.shape), '| device:', device)


[INFO] STL-10 features: (8000, 2048) | labels: (8000,) | device: cpu


In [2]:
def run_setting(name, k0, enable_split, enable_merge, lambda_param=2.0, epochs=50, warmup_epochs=20):
    k_star, acc, nmi, ari = train_clustering(
        features=X,
        labels=y,
        k_max=k0,
        device=device,
        epochs=epochs,
        lr=1e-3,
        lambda_param=lambda_param,
        gamma=0.1,
        warmup_epochs=warmup_epochs,
        enable_split=enable_split,
        enable_merge=enable_merge,
    )
    return {
        'Setting': name,
        'K0': k0,
        'K*': int(k_star),
        'ACC(%)': acc * 100,
        'NMI(%)': nmi * 100,
        'ARI(%)': ari * 100,
    }


In [3]:
results = [
    run_setting('PnP dynamic K', k0=20, enable_split=True, enable_merge=True),
    run_setting('Fixed-K baseline (wrong K=20)', k0=20, enable_split=False, enable_merge=False),
    run_setting('Fixed-K baseline (oracle K=10)', k0=10, enable_split=False, enable_merge=False),
]
main_df = pd.DataFrame(results)
main_df


,Setting,K0,K*,ACC(%),NMI(%),ARI(%)
0,PnP dynamic K,20,12,71.75,71.096525,57.847123
1,Fixed-K baseline (wrong K=20),20,20,48.45,63.092196,42.444422
2,Fixed-K baseline (oracle K=10),10,10,49.00,50.786451,31.943790


## Relative deviation

Bảng dưới đây để nhóm điền/reference so sánh với số liệu paper hoặc số liệu đối chiếu mà nhóm chọn cho STL-10.


In [4]:
reference_acc = {
    'PnP dynamic K': None,
    'Fixed-K baseline (wrong K=20)': None,
}

comparison_rows = []
for _, row in main_df.iterrows():
    ref = reference_acc.get(row['Setting'])
    if ref is None:
        continue
    reproduced = row['ACC(%)']
    relative_dev = abs(reproduced - ref) / max(abs(ref), 1e-8) * 100
    comparison_rows.append({
        'Setting': row['Setting'],
        'Reference ACC(%)': ref,
        'Reproduced ACC(%)': reproduced,
        'Relative Deviation(%)': relative_dev,
    })

pd.DataFrame(comparison_rows)


""
